# Ames Housing Price Prediction

## Project
**Objective:** The goal of this project is to predict the sale price of residential properties in Ames, Iowa, using the well-known Ames Housing Dataset. 

**Challenges:** 
1. **High Dimensionality:** Over 80 features.
2. **Data Quality:** The dataset contains missing values and outliers.
3. **Non-Linearity:** Real estate pricing often follows non-linear patterns and many features are highly skewd.
 
**Workflow:** 
EDA -> Preprocessing -> Modeling -> Evaluation


In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.linear_model import Ridge, LinearRegression, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

from sklearn import set_config
set_config(transform_output="pandas")

## 1. Data Loading & Initial Cleaning

In [ ]:
house_df = pd.read_csv('AmesHousing.csv')

house_df.head()

In [ ]:
# Cleaning column names
house_df.columns = house_df.columns.str.replace(' ', '')
house_df.rename(columns={'YearRemod/Add': 'YearRemodAdd'}, inplace=True)

In [ ]:
# Split Data
train, test = train_test_split(house_df, test_size=0.2, random_state=42)

trainX = train.drop(columns = ["SalePrice"])
trainY = np.log1p(train["SalePrice"]) # Log-transforming target for better normality

testX = test.drop(columns = ["SalePrice"])
testY = np.log1p(test["SalePrice"])

In [ ]:
# Create a working copy for exploration
df = trainX.copy()

## 2. Exploratory Data Analysis

In [ ]:
# Dataset Shape
r, c = df.shape
print(f"Dataset contains:\nRows: {r}\nColumns: {c}")

### Profiling the Data
Create a metadata table to see exactly what I'm dealing.


In [ ]:
profile = pd.DataFrame(index=df.columns)
profile["dtype"] = df.dtypes
profile["missing_count"] = df.isna().sum()
profile["missing_count_pct"] = df.isna().mean() * 100
profile["nunique"] = df.nunique()
profile["unique_ratio"] = df.nunique() / len(df)
profile["skew"] = df.select_dtypes(include="number").skew()

# Merge with descriptive stats
profile = profile.join(df.describe(include="all").T)
profile.drop(columns="unique", inplace=True, errors='ignore')
profile.head(10)

In [ ]:
# Numerical Columns Check
profile[
    profile["dtype"].isin([np.dtype("int64"), np.dtype("float64")])]

Features "MSSubClass", "OverallQual", "OverallCond" are Categorical.    
"MSSubClass" is nominal and the rest ordinal.

No impossible values

In [ ]:
# Categorical Columns Check
for col in df.columns:
    if profile.loc[col]["dtype"] == "str":
        print("\n")
        print(df[col].value_counts(dropna = False))

Feature Categories ok. No typos/repeated categories.

### Identifying Problematic Features
Flag columns that might need dropping or special handling.


In [ ]:
profile["high_missing"] = profile["missing_count_pct"] > 0 
profile["id_like"] = profile["unique_ratio"] > 0.95 
profile["constant"] = profile["nunique"] <= 1
profile["high_skew"] = profile["skew"].abs() > 1

In [ ]:
# Check ID-like features 
print("ID-like features found:", profile.index[profile["id_like"]].tolist())

Drop 'Order' & 'PID

In [ ]:
# Exploring Missing Value handling
profile.loc[profile["high_missing"] == True]

Missing values at most cases means 'Feature doesnt exist in the house' (eg. Garage).    
'LotFrontage' need extra investigation.

### Investigation: Lot Frontage
Is LotFrontage missing at random? Let's check if it varies by neighborhood.

In [ ]:
frontage_check = df.groupby("Neighborhood")["LotFrontage"].agg(["median", "mean", "std"])
print(frontage_check.head())

The median varies significantly by neighborhood.        
Impute LotFrontage using Neighborhood median rather than a global mean.


### Investigation: Basement Consistency


In [ ]:
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
for col in bsmt_cols:
    print(f"{col}: {len(df.loc[df[col].isna()])}")

df[bsmt_cols].loc[df["BsmtQual"].isna()]

Most basement features are missing together, implying the house simply has no basement.


### Investigation: Outliers


In [ ]:
summary = []
for col in df.select_dtypes(include="number").columns:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))]
    summary.append({"feature": col, "outlier_count": len(outliers), "skew": df[col].skew()})

summary_df = pd.DataFrame(summary).sort_values("outlier_count", ascending=False)
summary_df.head(10)

Data has a lot of outliers but part of it is that features ar just absent from many houses.
Also, features like 'LotArea' are inheritly skewed, making them have outliers.
For now i'll use LogTransform to midigate the effect on the models.

## 3. Feature Engineering & Pipeline Construction


In [ ]:
# Grouping columns for the Pipeline
none_cols = ["Alley", "MasVnrType", "FireplaceQu", "BsmtQual", "BsmtCond", "BsmtExposure", 
             "BsmtFinType1", "BsmtFinType2", "GarageType", "GarageFinish", "GarageQual", 
             "GarageCond", "PoolQC", "Fence", "MiscFeature"]

zero_cols = ["MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
             "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "Fireplaces", "BsmtFullBath", 
             "BsmtHalfBath", "FullBath", "HalfBath", "GarageCars", "GarageArea",
             "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
             "ScreenPorch", "PoolArea", "MiscVal"]

mode_cols = ["MoSold", "MSZoning", "Street", "LandContour", "LotConfig", "Neighborhood",
             "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",
             "Exterior1st", "Exterior2nd", "Foundation", "Heating", "CentralAir",
             "Functional", "PavedDrive", "SaleType", "SaleCondition", "MSSubClass", 
             "LotShape", "Utilities", "LandSlope", "Electrical", "ExterQual", "ExterCond", 
             "HeatingQC", "KitchenQual", "OverallQual", "OverallCond"]

median_cols = ["LotArea", "BedroomAbvGr", "KitchenAbvGr", "TotRmsAbvGrd", "GrLivArea"]

### Custom Transformers

In [ ]:
class NeighborhoodMedianImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.medians_ = X.groupby("Neighborhood")["LotFrontage"].median()
        self.global_median_ = X["LotFrontage"].median()
        return self
    def transform(self, X):
        X = X.copy()
        X["LotFrontage"] = X["LotFrontage"].fillna(X["Neighborhood"].map(self.medians_))
        X["LotFrontage"] = X["LotFrontage"].fillna(self.global_median_)
        return X

class YearBuiltImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X["YearRemodAdd"] = X["YearRemodAdd"].fillna(X["YearBuilt"])
        X["GarageYrBlt"] = X["GarageYrBlt"].fillna(X["YearBuilt"])
        return X

class AgeFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
        X["RemodelAge"] = X["YrSold"] - X["YearRemodAdd"]
        X["GarageAge"] = X["YrSold"] - X["GarageYrBlt"]
        return X

class SkewLogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=1): self.threshold = threshold
    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include="number").columns
        skewness = X[num_cols].skew()
        self.skewed_cols = skewness[skewness.abs() > self.threshold].index.tolist()
        return self
    def transform(self, X):
        X = X.copy()
        X[self.skewed_cols] = np.log1p(X[self.skewed_cols])
        return X

## 4. Training and Evaluation
I will now run a comparison across multiple models. 

In [ ]:
# Numerical Column
num_cols = ["LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1",
            "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "1stFlrSF", 
            "2ndFlrSF", "LowQualFinSF", "GrLivArea", "BsmtFullBath", 
            "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr", 
            "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageCars",
            "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", 
            "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal",
            "HouseAge", "RemodelAge", "GarageAge"]

# One Hot Encode
nominal_cols = ["MSZoning", "Alley", "LandContour", 
                "LotConfig", "Neighborhood", "Condition1", "Condition2", 
                  "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",
                  "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation", 
                  "Heating", "Functional", "GarageType", 
                  "GarageFinish", "PavedDrive", "MiscFeature", "SaleType", 
                  "SaleCondition", "MSSubClass"]

# Ordinal Encode
ordinal_cols = ["LotShape", "Utilities", "LandSlope", "ExterQual", 
                "ExterCond", "BsmtQual", "BsmtCond", "BsmtExposure", 
                  "BsmtFinType1", "BsmtFinType2", "HeatingQC", "Electrical", 
                  "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", 
                  "PoolQC", "Fence", "OverallQual", "OverallCond"]

ordinal_mappings = {
            "LotShape": {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
            "Utilities": {"ELO": 1, "NoSeWa": 2, "NoSewr": 3, "AllPub": 4},
            "LandSlope": {"Sev": 1, "Mod": 2, "Gtl": 3},
            "ExterQual": {"Po": 1, "Fa": 2, "Gd":3, "Ex": 4},
            "ExterCond": {"Po": 1, "Fa": 2, "Gd":3, "Ex": 4},
            "BsmtQual": {"None": 1, "Po": 2, "Fa": 3, "Gd":4, "Ex": 5},
            "BsmtCond": {"None": 1, "Po": 2, "Fa": 3, "Gd":4, "Ex": 5},
            "BsmtExposure": {"None": 1, "No":2, "Mn":3, "Av":4, "Gd": 5},
            "BsmtFinType1": {"None": 1, "Unf": 2, "LwQ": 3, "Rec": 4, "BLQ": 5, "ALQ": 6, "GLQ": 7},
            "BsmtFinType2": {"None": 1, "Unf": 2, "LwQ": 3, "Rec": 4, "BLQ": 5, "ALQ": 6, "GLQ": 7},
            "HeatingQC": {"Po": 1, "Fa": 2, "Gd":3, "Ex": 4},
            "Electrical": {"Mix": 1, "FuseP": 2, "FuseF": 3, "FuseA": 4, "SBrkr": 5},
            "KitchenQual": {"Po": 1, "Fa": 2, "Gd":3, "Ex": 4},
            "FireplaceQu": {"None": 1, "Po": 2, "Fa": 3, "TA": 4, "Gd":5, "Ex": 6},
            "GarageQual": {"None": 1, "Po": 2, "Fa": 3, "TA": 4, "Gd":5, "Ex": 6},
            "GarageCond": {"None": 1, "Po": 2, "Fa": 3, "TA": 4, "Gd":5, "Ex": 6},
            "PoolQC": {"None": 1, "Fa": 2, "TA": 3, "Gd":4, "Ex": 5},
            "Fence": {"None": 1, "MnWw": 2, "GdWo": 3, "MnPr": 4, "GdPr": 5}, 		
            "OverallQual": {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10},
            "OverallCond": {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10}
            }

class OrdinalMapper(BaseEstimator, TransformerMixin):

    def __init__(self, mappings):
        self.mappings = mappings

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        for col, mapping in self.mappings.items():
            if col in X.columns:
                X[col] = X[col].map(mapping)
            
            X[col] = X[col].fillna(-1)

        return X

In [ ]:
# Define Encoding and Scaling
num_cols_final = median_cols + zero_cols + ["LotFrontage", "HouseAge", "RemodelAge", "GarageAge"]
nominal_cols = ["MSZoning", "Neighborhood", "Foundation", "GarageType", "SaleType", "SaleCondition"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal_cols),
        ("ord", OrdinalMapper(mappings=ordinal_mappings), ordinal_cols)        
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

In [ ]:
models = {
    "Ridge": Ridge(alpha=10.0),
    "LinearRegression": LinearRegression(),
    "Lasso": Lasso(alpha=1, max_iter = 10000),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoost": GradientBoostingRegressor(n_estimators=1000, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42),
    "NeuralNetwork": MLPRegressor(hidden_layer_sizes=(100, 100), max_iter=1000, random_state=42)
}

results = []

for name, model in models.items():
    # Constructing the full pipeline
    full_pipeline = Pipeline([
        ("lotfrontage_imp", NeighborhoodMedianImputer()),
        ("yrbuilt_imp", YearBuiltImputer()),
        ("impute_others", ColumnTransformer([
            ("none", SimpleImputer(strategy="constant", fill_value="None"), none_cols),
            ("zero", SimpleImputer(strategy="constant", fill_value=0), zero_cols),
            ("mode", SimpleImputer(strategy="most_frequent"), mode_cols),
            ("median", SimpleImputer(strategy="median"), median_cols)
        ], remainder="passthrough", verbose_feature_names_out=False)),
        ("age_eng", AgeFeatureEngineer()),
        ("log_skew", SkewLogTransformer(threshold=1)),
        ("preprocess", preprocessor),
        ("regressor", model)
    ])
    
    full_pipeline.fit(trainX, trainY)
    preds = full_pipeline.predict(testX)
    
    # Metrics
    mae = mean_absolute_error(testY, preds)
    rmse = root_mean_squared_error(testY, preds)
    results.append({"Model": name, "MAE": mae, "RMSE": rmse})

In [ ]:
print(trainY[:5])

In [ ]:
# Baseline model (MEAN Predictor)
mean_value = np.mean(trainY)

y_pred_baseline = np.full(shape=len(testY), fill_value=mean_value)

# 3. evaluate
mae = mean_absolute_error(testY, y_pred_baseline)
rmse = root_mean_squared_error(testY, y_pred_baseline, squared=False)

results.append({"Model": "Baseline Mean Predictor", "MAE": mae, "RMSE": rmse})

## 5. Final Results & Observations

In [ ]:
results_df = pd.DataFrame(results).sort_values("MAE")
print(results_df)

### Summary of the Process:
1. **EDA:** Investigated basement features and LotFrontage. Decided on neighborhood-based imputation. Checked missing values, outliers and all features for impossible values or wrong inputs.
2. **Preprocessing:** Remove "id-like" columns(not use them during modeling). Impute LotArea with Neighborhood Median and also the rest of missing values(absent of features in house).
3. **Feature Engineering:** Create features like "HouseAge".
4. **Handling Skewness:** Rather than deleting outliers, I used a Log Transformation to allow the model to learn from them without being overwhelmed by their scale.
5. **Encoding:** 
5. **Model Performance:** 
    The best performing model was **XGBoost**, achieving a **MAE** of 0.0739 in log space, corresponding to approximately 7–8% average prediction error. Gradient boosting methods consistently outperformed other approaches, confirming their suitability for structured tabular regression tasks.

    Interestingly, linear models (Linear Regression and Ridge) performed nearly on par with boosting methods, suggesting that the feature engineering process effectively linearized much of the underlying signal in the dataset.

    In contrast, neural networks significantly underperformed, highlighting their limited effectiveness on small-scale tabular datasets. Lasso regression also performed poorly.

In [ ]:
# Visualizing Neural Network predictions
plt.figure(figsize=(8, 6))
plt.scatter(testY, preds, alpha=0.4, color='darkblue')
plt.plot([testY.min(), testY.max()], [testY.min(), testY.max()], '--r', linewidth=2)
plt.title("Actual vs Predicted Housing Prices (Log Scale)")
plt.xlabel("Actual Log Sale Price")
plt.ylabel("Predicted Log Sale Price")
plt.show()